In [1]:
import pandas as pd
import numpy as np


In [2]:
FEATURE_PATH = "../Data/Feature_Data"

enrol_feat = pd.read_csv(
    f"{FEATURE_PATH}/feature_enrolment.csv",
    parse_dates=["date"]
)

demo_bio_feat = pd.read_csv(
    f"{FEATURE_PATH}/feature_demo_bio_combined.csv",
    parse_dates=["date"]
)


In [3]:
sort_keys = ["state_clean", "district_clean", "pincode", "date"]

enrol_feat = enrol_feat.sort_values(sort_keys)
demo_bio_feat = demo_bio_feat.sort_values(sort_keys)


In [4]:
enrol_feat["enrolment_dod_change"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .diff()
)

enrol_feat["enrolment_dod_pct_change"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .pct_change()
)


In [5]:
enrol_feat["enrolment_7day_mean"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

enrol_feat["enrolment_7day_std"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .transform(lambda x: x.rolling(7, min_periods=1).std())
)


In [6]:
enrol_feat["enrolment_volatility"] = np.where(
    enrol_feat["enrolment_7day_mean"] > 0,
    enrol_feat["enrolment_7day_std"] / enrol_feat["enrolment_7day_mean"],
    0
)


In [7]:
enrol_feat["sudden_surge_flag"] = (
    enrol_feat["enrolment_dod_pct_change"] > 0.5
).astype(int)

enrol_feat["sudden_drop_flag"] = (
    enrol_feat["enrolment_dod_pct_change"] < -0.5
).astype(int)


In [8]:
demo_bio_feat["bio_demo_stress_ratio"] = np.where(
    demo_bio_feat["total_demographic_updates"] > 0,
    demo_bio_feat["total_biometric_updates"] /
    demo_bio_feat["total_demographic_updates"],
    0
)


In [9]:
demo_bio_feat["high_biometric_stress_flag"] = (
    demo_bio_feat["bio_demo_stress_ratio"] > 1.5
).astype(int)


In [10]:
import os

ADV_FEATURE_PATH = "../Data/Advanced_Feature_Data"
os.makedirs(ADV_FEATURE_PATH, exist_ok=True)


In [11]:
enrol_feat.to_csv(
    f"{ADV_FEATURE_PATH}/advanced_feature_enrolment.csv",
    index=False
)

demo_bio_feat.to_csv(
    f"{ADV_FEATURE_PATH}/advanced_feature_demo_bio.csv",
    index=False
)


In [12]:
import os
os.listdir("../Data/Advanced_Feature_Data")


['advanced_feature_demo_bio.csv', 'advanced_feature_enrolment.csv']